# ADALL Merged Notebook: Data Prep → Modelling → Evaluation → Revision

**This notebook merges three separate class notebooks into one runnable Colab file:**

| Part | Source | Dataset | What it covers |
|---|---|---|---|
| **Part A** | Week 3 | Laptop prices (SGD) | Business framing, LLM-assisted data quality review, cleaning, baseline model |
| **Part B** | Week 4 | Laptop prices (SGD) — continues Part A | Multi-model comparison, cross-validation, error diagnosis, feature importance |
| **Part C** | Week 5 | Student performance (failed subjects) | A second, self-contained regression workflow used for practical-test revision, with emphasis on **leakage** |

## How to read this notebook
Every code cell that calls an LLM is preceded by a markdown cell titled **"Why this prompt is used"**.
These notes exist because the point of this course is not the Python syntax — it's **prompt design**:
what information you choose to give the model, what you deliberately withhold, and what constraints
you put on its output so the response is usable rather than generic.

Every modelling cell is preceded by a short note on **why that model** was chosen at that point in the
workflow (baseline vs. comparison vs. tuned), and followed by a note on **how to interpret the output**
(what a good number looks like, what a red flag looks like).

## Model used for LLM calls
All prompt cells below use the OpenAI Responses API with `gpt-5.4-nano`, matching the original
notebooks. This is a small, fast, inexpensive model — appropriate here because the tasks are narrow
(reviewing a text summary, suggesting features, drafting a paragraph), not open-ended reasoning.
Using a bigger/pricier model would not meaningfully improve these specific tasks, and using a
smaller/cheaper model risks vaguer, less-grounded answers. If you don't have an API key, set
`RUN_API_CELLS = False` in the setup cells and copy-paste the printed prompt into any chatbot instead —
the notebook is designed to work either way.

## Running this in Colab
1. Upload this `.ipynb` to Google Colab (or open directly from Drive).
2. If you want live LLM calls, add `OPENAI_API_KEY` to Colab **Secrets** (key icon in the left sidebar).
3. Run cells top to bottom. Part A must run before Part B (Part B reuses Part A's cleaned data in memory).
   Part C is independent and can be run on its own.


---
# PART A — Preparing Data and Building a Baseline Model
*(merged from Week 3)*

**Business problem:** A refurbished laptop seller wants to price laptops fairly and consistently.
**Modelling objective:** predict `Price_SGD` from laptop specifications — a **regression** task, because
the target is a continuous dollar amount, not a category.


## A1. Setup

**Why these libraries:** `pandas`/`numpy` for data handling, `matplotlib` for the plots we need later
to *see* model errors (a table of numbers hides patterns a scatter plot reveals), and
`sklearn` pipeline/preprocessing/model classes so training and evaluation stay consistent.
`joblib` lets us persist the trained model object if needed.


In [ ]:
# Libraries used in this notebook
import os
import shutil
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# Modelling libraries
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.inspection import permutation_importance

!pip -q install xgboost
from xgboost import XGBRegressor

pd.set_option('display.max_columns', 100)
RANDOM_STATE = 42
print('Setup complete')

## A2. Load the dataset

We load directly from the GitHub raw CSV link used in class. **Why a raw link and not the GitHub
preview page:** the preview page is HTML, not CSV — `pd.read_csv()` needs the raw file bytes.


In [ ]:
github_raw_url = 'https://raw.githubusercontent.com/rq-goh/ADALL_github/refs/heads/main/laptop_prices_2024_sgd_TL.csv'

df = pd.read_csv(github_raw_url)

print('Dataset loaded successfully.')
print('Shape:', df.shape)
display(df.head())

## A3. Quick manual inspection *before* touching an LLM

**Why we look first, before prompting:** an LLM can only reason about what you show it. If you don't
know your own dataset's shape and columns, you can't judge whether the LLM's answer is sensible — you'd
just be trusting it blindly. This is a deliberate step, not busywork.


In [ ]:
print('Shape:', df.shape)
display(df.head())
df.info()
print(df.columns.tolist())

## A4. First LLM touchpoint — a small 10-row preview

**Why start with only 10 rows, not the full dataset:** it's the smallest, cheapest thing that could
possibly work. A 10-row preview costs almost nothing to send and is enough for the model to guess at
column meaning and the likely target. We only escalate to a fuller summary (the "payload text" in A6)
once we've established the preview alone isn't representative enough.

**Boundary to note:** building `data_preview` below does *not* send anything anywhere — it's a local
Python string. Data is only transmitted the moment we call `client.responses.create(...)`.


In [ ]:
data_preview = df.head(10).to_string()
print(data_preview[:1500])

### Set up the API connection

`RUN_API_CELLS` is a manual on/off switch. **Why default to a switch rather than always calling the
API:** it lets the notebook run end-to-end even without a key (you just copy the printed prompt into
a chatbot), so the notebook isn't broken by a missing credential, and it stops accidental repeated API
spend while iterating on a prompt.


In [ ]:
RUN_API_CELLS = True
OPENAI_MODEL = 'gpt-5.4-nano'
client = None

if RUN_API_CELLS == True:
    from google.colab import userdata
    from openai import OpenAI
    api_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=api_key)
    print('OpenAI client is ready.')
else:
    print('Manual chatbot mode. Copy the prompts when they appear.')

### Why this prompt is used — preview review

The prompt below deliberately:
- **Gives only the preview text**, not the full dataframe, so we can test whether 10 rows is
  "enough" — this is the experiment, not just a shortcut.
- **Asks three narrow, closed questions** (what does a row represent / what's the likely target /
  what checks should we run) instead of an open "tell me about this dataset". A vague prompt invites a
  vague, generic-sounding answer; a narrow prompt forces the model to commit to specific, checkable
  claims we can verify against the dataframe ourselves.
- **Explicitly asks for a short, practical answer** — without this instruction, models tend to pad
  responses with disclaimers and general advice that isn't tied to *this* dataset.

**How to interpret the output:** treat the LLM's guess at the target column as a hypothesis, not a
fact — confirm it yourself by checking which column looks like a price/outcome variable.


In [ ]:
preview_prompt = f"""
Here are the first 10 rows of a laptop pricing dataset:

{data_preview}

Questions:
1. What does each row appear to represent?
2. Which column is likely the target for a price prediction model?
3. What are 3 possible data quality checks we should perform before modelling?

Keep the answer short and practical.
"""

print('Prompt to send:')
print(preview_prompt[:2000])

if client is not None:
    response = client.responses.create(model=OPENAI_MODEL, input=preview_prompt)
    print('\nLLM response:')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')

## A5–A6. From preview to payload text

**Why the preview alone is risky:** if the first 10 rows happen to all be gaming laptops, the LLM will
wrongly generalise "this dataset is mostly gaming laptops" — a preview has no guarantee of being
representative. So we build **payload text**: a compact, whole-dataset *statistical summary*
(shape, dtypes, numeric describe(), missing values, unique counts, correlations, top categorical
values, and simple leakage/ID warnings) instead of sending every row.

**Why payload text over sending the full dataset (the actual design decision here):**

| Approach | Strength | Weakness |
|---|---|---|
| Send 10 rows | cheap, easy to read | not representative |
| Send full dataset | complete | slow, costly in tokens, privacy/governance risk, may exceed context |
| **Send payload text** | balanced summary, covers every row statistically | LLM only sees aggregates, not individual records |

This is the central prompting idea of this course: you almost never need to hand a model your raw data —
a well-chosen *summary* is usually enough for the model to reason about readiness, and it's dramatically
cheaper and safer.


In [ ]:
payload_text = ''

# 1. Shape
payload_text += '=== SHAPE ===\n'
payload_text += 'Rows: ' + str(df.shape[0]) + '\n'
payload_text += 'Columns: ' + str(df.shape[1]) + '\n\n'

# 2. Column names and data types
payload_text += '=== COLUMNS AND DATA TYPES ===\n'
payload_text += df.dtypes.to_string()
payload_text += '\n\n'

# 3. Numeric summary
payload_text += '=== NUMERIC SUMMARY ===\n'
numeric_summary = df.describe(include='number').round(2)
payload_text += numeric_summary.to_string()
payload_text += '\n\n'

# 4. Missing values
payload_text += '=== MISSING VALUES ===\n'
missing_table = pd.DataFrame()
missing_table['missing_count'] = df.isna().sum()
missing_table['missing_pct'] = (df.isna().sum() / len(df) * 100).round(2)
payload_text += missing_table.to_string()
payload_text += '\n\n'

# 5. Unique values per column
payload_text += '=== UNIQUE VALUES PER COLUMN ===\n'
unique_table = pd.DataFrame()
unique_table['unique_count'] = df.nunique(dropna=False)
payload_text += unique_table.to_string()
payload_text += '\n\n'

# 6. Correlation between numeric columns
payload_text += '=== CORRELATION BETWEEN NUMERIC COLUMNS ===\n'
correlation_table = df.corr(numeric_only=True).round(2)
payload_text += correlation_table.to_string()
payload_text += '\n\n'

# 7. Top 10 values for categorical columns
payload_text += '=== TOP 10 VALUES FOR CATEGORICAL COLUMNS ===\n'
categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
if len(categorical_columns) == 0:
    payload_text += 'No categorical columns found.\n'
else:
    for col in categorical_columns:
        payload_text += '\nColumn: ' + col + '\n'
        payload_text += df[col].value_counts(dropna=False).head(10).to_string()
        payload_text += '\n'
    payload_text += '\n'

# 8. Simple warning checks
payload_text += '=== SIMPLE WARNING CHECKS ===\n'
id_like_columns = []
constant_columns = []
for col in df.columns:
    unique_count = df[col].nunique(dropna=False)
    if unique_count == len(df):
        id_like_columns.append(col)
    if unique_count <= 1:
        constant_columns.append(col)

payload_text += 'Possible ID-like columns: ' + str(id_like_columns) + '\n'
payload_text += 'Constant columns: ' + str(constant_columns) + '\n'
payload_text += 'Duplicate rows: ' + str(df.duplicated().sum()) + '\n'

print(payload_text)

### Why this prompt is used — data quality review from payload text

Compared to the preview prompt, this one adds explicit **guardrails**:

- *"Do not assume external knowledge"* — stops the model inventing facts about laptop brands/models
  that aren't in the data.
- *"Do not say to drop a column just because it is listed as a warning"* — this is the single most
  important instruction in the prompt. A warning (e.g. "this column is 100% unique") is a *flag to
  inspect*, not an automatic rule. Without this line, LLMs reliably over-recommend dropping columns
  just because a heuristic fired, which can silently remove real signal (e.g. a genuinely unique but
  informative product-model column).
- *"Keep the answer concise"* — because this output feeds into a human decision, not a report; a long
  answer is harder to act on, not more rigorous.

**How to interpret the output:** the response is a checklist of *candidates* to inspect (possible
leakage, ID-like columns, high-cardinality columns) — every one of those still needs a human decision
using domain knowledge, which is exactly what the cleaning step below does.


In [ ]:
quality_prompt = f"""
You are helping a data analytics student prepare a laptop pricing dataset for modelling.

Dataset profile:
{payload_text}

Task:
1. List all data quality or modelling-readiness issues.
2. Suggest action to remedy the issues.

Important:
- Do not assume external knowledge.
- Do not say to drop a column just because it is listed as a warning.
- Keep the answer concise.
"""

if client is not None:
    response = client.responses.create(model=OPENAI_MODEL, input=quality_prompt)
    print('\nLLM response:')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')

## A7. Apply a cleaning plan

We now act on the LLM's suggestions, but **only after human judgement**. Below is a worked example that:
1. reverses discount columns to reconstruct an "original" price (fixing possible leakage where
   `Price_SGD` was partly *derived from* discount fields — i.e. a feature was accidentally encoding the
   answer),
2. drops the now-redundant discount columns,
3. drops obvious index/ID-like columns,
4. drops constant (zero-variance) columns.

**Why not just run whatever the LLM outputs verbatim:** an LLM cannot see your actual dataframe values
at cell-execution time — it only reasoned from the payload text summary. Each step here is kept
because it survived a sanity check against the real data (e.g. we confirm `Unnamed: 0` really is
100% unique and non-null before dropping it), not because the model said so.


In [ ]:
cleaned_df = df.copy()

# 1. Reverse the discounts to estimate the original price (fixes possible leakage)
if {'Price_SGD', 'Member_Discount', 'Brand_Discount', 'Discount_percent'}.issubset(cleaned_df.columns):
    price_after_fixed_discounts = (
        cleaned_df['Price_SGD']
        + cleaned_df['Member_Discount']
        + cleaned_df['Brand_Discount']
    )
    discount_multiplier = 1 - (cleaned_df['Discount_percent'] / 100)
    cleaned_df['Price_SGD'] = (price_after_fixed_discounts / discount_multiplier).round(2)
    print('Step 1: Price_SGD adjusted to estimated original price.')

    # 2. Drop the original discount columns to prevent target leakage/dominance
    columns_to_drop = ['Member_Discount', 'Brand_Discount', 'Discount_percent']
    cleaned_df = cleaned_df.drop(columns=columns_to_drop)
    print('Step 2: Discount columns dropped.')
else:
    print('Discount columns not found as expected — skipping discount-reversal step.')

# 3. Drop ID-like columns
unnamed_cols = [col for col in cleaned_df.columns if 'unnamed' in col.lower()]
cleaned_df = cleaned_df.drop(columns=unnamed_cols)
print(f'Step 3: ID-like columns dropped: {unnamed_cols}')

# 4. Drop constant columns
constant_cols = [col for col in cleaned_df.columns if cleaned_df[col].nunique(dropna=False) <= 1]
cleaned_df = cleaned_df.drop(columns=constant_cols)
print(f'Step 4: Constant columns dropped: {constant_cols}')

print(f'\nOriginal shape: {df.shape}')
print(f'Cleaned shape: {cleaned_df.shape}')
display(cleaned_df.head())

In [ ]:
print('Original rows:', df.shape[0])
print('Cleaned rows:', cleaned_df.shape[0])
print('\nMissing values after cleaning:')
display(cleaned_df.isna().sum().sort_values(ascending=False).head(10))
print('\nColumns after cleaning:')
print(cleaned_df.columns.tolist())

## A8. Split into X and y, train/test split

**Why `X` must never contain `Price_SGD`:** if the target leaks into the inputs, the model doesn't
learn to predict price — it learns to look up the answer, which produces a misleadingly perfect score
that collapses the moment it meets genuinely unseen data.

**Why a held-out test set at all:** training performance always looks better than real-world
performance, because the model has literally seen those rows before. The test set is the model's first
honest exam.


In [ ]:
target_col = 'Price_SGD'

if target_col not in cleaned_df.columns:
    raise ValueError(f'Target column {target_col} was not found. Check your cleaned dataset columns.')

X = cleaned_df.drop(columns=[target_col])
y = cleaned_df[target_col]

print('X shape:', X.shape)
print('y shape:', y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print('Training rows:', X_train.shape[0])
print('Test rows:', X_test.shape[0])

## A9. Preprocessing pipeline and baseline model

**Why a `Pipeline` and not manual transforms:** it guarantees the exact same preprocessing (fit only on
train, applied identically to test) — a common beginner bug is fitting an encoder on the full dataset
before splitting, which silently leaks test-set information into training.

**Why a Decision Tree as the *baseline* model specifically:** it is fast, easy to explain to a
non-technical stakeholder ("if RAM is above X and brand is Y, price tends to be Z"), and needs almost
no tuning. The point of a baseline is not to be the best model — it's to give every later model
(Random Forest, XGBoost) a bar it must clear to justify its added complexity.


In [ ]:
cat_columns = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_columns = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

print('Categorical columns:', cat_columns)
print('Numeric columns:', num_columns)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_columns),
        ('num', StandardScaler(), num_columns)
    ],
    remainder='drop'
)

baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=5))
])

baseline_model.fit(X_train, y_train)
print('Baseline model trained.')

## A10. Evaluate the baseline

**Metric choice — why MAE is the headline number:** MAE is in the same unit as the target (SGD), so
"MAE = $120" is instantly meaningful to a non-technical stakeholder ("on average we're off by $120").
RMSE is reported alongside because it penalises large errors more, which matters if a few very wrong
predictions are more costly than many small ones. R² is reported last because, while common, it's the
hardest of the three to explain to a business audience.

**How to interpret the output:** compare MAE against the typical laptop price — an MAE that's a small
fraction of the average price is encouraging; an MAE close to the price range itself means the model
isn't adding much value yet.


In [ ]:
y_pred = baseline_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

baseline_results = pd.DataFrame({
    'Model': ['Baseline Decision Tree'],
    'MAE': [mae],
    'RMSE': [rmse],
    'R2': [r2]
})
display(baseline_results.round(3))

**Checkpoint for Part A:** we now have a cleaned dataset (`cleaned_df`), a leak-free `X`/`y` split,
and a baseline model score to beat. Part B continues directly from these in-memory objects — in the
original two-notebook version, these were saved to CSV and reloaded from GitHub; here we skip that
round-trip since everything is already in memory.


---
# PART B — Modelling and Evaluation
*(merged from Week 4 — continues directly from Part A's `X_train`, `X_test`, `y_train`, `y_test`)*

**Goal of this part:** move from "one plausible model" to "a defensible choice of model", using
cross-validation, a sanity-check baseline, and several kinds of error diagnosis — not just a single
accuracy number.


## B1. Confirm the modelling task

Before writing any modelling code, restate the task in one line. This matters because it's exactly the
kind of framing question a vibe-coded LLM prompt (see below) tends to get wrong if you don't specify it.

| Item | Decision |
|---|---|
| Task type | Regression |
| Target | `Price_SGD` |
| Main metric | MAE — easy to explain in dollars |


In [ ]:
print('Training columns:')
print(X_train.columns.tolist())
print('Target summary:')
print(y_train.describe())

## B2. Why *not* to just vibe-code the modelling step

**Why this comparison is included deliberately:** a common student instinct is to type a vague prompt
like *"perform modelling with X_train, X_test, y_train, and y_test"* and run whatever comes back. Try
this yourself in Gemini or ChatGPT and check: did it one-hot encode categoricals? Did it use a
pipeline? Did it use cross-validation, or just one score? Did it assume classification instead of
regression? A vague prompt usually fails at least one of these — and you'd only notice if you already
know what "good" modelling code looks like, which is why Part A and B build that manually first.

### Why this prompt is used — a *better* version of the same request

Compare the vague prompt above to the one actually used below. The improved version explicitly names:
- the three models to compare (so the LLM doesn't default to one arbitrary choice),
- "perform one-hot encoding" and "use pipeline" (removes ambiguity about preprocessing),
- cross-validation (removes ambiguity about how performance should be measured).

Even with these improvements, note the caveat below the cell: OpenAI's response may still add
unnecessary machinery (e.g. `GridSearchCV`) that's overkill for a small, simple dataset — a reminder
that a well-specified prompt narrows the *plausible range* of outputs, but doesn't guarantee an
appropriately-scoped one. You still have to review the output.


In [ ]:
quality_prompt2 = f"""
Generate python code to perform regression modelling with dtr, rfr, xgb. perform one-hot encoding, use pipeline and cross validation.
"""

if client is not None:
    response2 = client.responses.create(model=OPENAI_MODEL, input=quality_prompt2)
    print('\nLLM response:')
    print(response2.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')

## B3. Manual modelling — the version we actually trust

We build this by hand (not from the LLM output above) so we know exactly what preprocessing and
comparison logic is being used.


In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print('Numeric columns:', num_cols)
print('Categorical columns:', cat_cols)

preprocessor_b = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)
preprocessor_b

## B4. Three models, same preprocessing, same split

**Why these three specific models, in this order:**

| Model | Why it's included |
|---|---|
| **Decision Tree** | The simplest tree model — a repeat of the Part A baseline, for a fair comparison point. |
| **Random Forest** | Averages many decision trees over different data/feature subsets, usually reducing variance (overfitting) versus a single tree, at some cost in interpretability. |
| **XGBoost** | Builds trees sequentially, each correcting the previous trees' mistakes. Often the strongest of the three on tabular data, but also the easiest to over-tune — kept deliberately simple here (`max_depth=4`, modest `learning_rate`) rather than heavily tuned. |

Comparing all three under **identical** preprocessing and split is the point — any performance
difference should come from the model, not from an accidental difference in how the data was prepared.


In [ ]:
dt_model = Pipeline(steps=[
    ('preprocessor', preprocessor_b),
    ('model', DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE))
])
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
dt_mae = mean_absolute_error(y_test, dt_pred)
dt_rmse = mean_squared_error(y_test, dt_pred) ** 0.5
dt_r2 = r2_score(y_test, dt_pred)
print('Decision Tree MAE :', round(dt_mae, 2))
print('Decision Tree RMSE:', round(dt_rmse, 2))
print('Decision Tree R2  :', round(dt_r2, 3))

In [ ]:
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor_b),
    ('model', RandomForestRegressor(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1))
])
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = mean_squared_error(y_test, rf_pred) ** 0.5
rf_r2 = r2_score(y_test, rf_pred)
print('Random Forest MAE :', round(rf_mae, 2))
print('Random Forest RMSE:', round(rf_rmse, 2))
print('Random Forest R2  :', round(rf_r2, 3))

In [ ]:
xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor_b),
    ('model', XGBRegressor(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        objective='reg:squarederror',
        n_jobs=1
    ))
])
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = mean_squared_error(y_test, xgb_pred) ** 0.5
xgb_r2 = r2_score(y_test, xgb_pred)
print('XGBoost MAE :', round(xgb_mae, 2))
print('XGBoost RMSE:', round(xgb_rmse, 2))
print('XGBoost R2  :', round(xgb_r2, 3))

**How to interpret this table:** lower MAE/RMSE and higher R² are better, but don't stop here —
one test-set score can be a lucky or unlucky split. Cross-validation (next) checks whether the ranking
holds up across several different splits.


In [ ]:
test_results = pd.DataFrame({
    'model': ['Decision Tree', 'Random Forest', 'XGBoost'],
    'test_mae': [dt_mae, rf_mae, xgb_mae],
    'test_rmse': [dt_rmse, rf_rmse, xgb_rmse],
    'test_r2': [dt_r2, rf_r2, xgb_r2]
})
test_results = test_results.sort_values('test_mae')
test_results

## B5. Cross-validation check

**Why ShuffleSplit with 5 rounds:** a single train/test split is one sample of "how well does this
model generalise" — it could be unusually easy or unusually hard by chance. `ShuffleSplit` repeats the
train/validate process 5 times on different random subsets of the *training* data (the test set stays
untouched), so `cv_mae_mean` is a much more stable estimate of true generalisation error than any one
split, and `cv_mae_std` tells you how much that estimate wobbles.

**How to interpret this table:** prefer the model with the lowest `cv_mae` — but if two models are very
close, prefer the simpler one (Decision Tree > Random Forest > XGBoost in interpretability), since
added complexity should earn its place with a meaningfully better score, not just a marginally better one.


In [ ]:
cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=RANDOM_STATE)

dt_cv_scores = cross_val_score(dt_model, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error')
rf_cv_scores = cross_val_score(rf_model, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error')
xgb_cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error')

cv_results = pd.DataFrame({
    'model': ['Decision Tree', 'Random Forest', 'XGBoost'],
    'cv_mae': [-dt_cv_scores.mean(), -rf_cv_scores.mean(), -xgb_cv_scores.mean()],
    'cv_mae_std': [dt_cv_scores.std(), rf_cv_scores.std(), xgb_cv_scores.std()]
})
cv_results = cv_results.sort_values('cv_mae')
cv_results

# Note: cross_val_score maximises scores by convention, but MAE is an error (lower = better),
# so scikit-learn returns it negative. We negate it back (-scores) to read it as normal, positive MAE.

### Why this prompt is used — a stricter follow-up vibe-code request

This second version adds `objective='reg:squarederror'` and cross-validation explicitly, closing the
gaps found in the first vibe-coded attempt (B2). Comparing the two prompt outputs side by side is the
actual teaching point: **prompt specificity directly determines whether the generated code matches the
standard you've built manually.**


In [ ]:
quality_prompt3 = f"""
Generate python code to perform regression modelling with dtr, rfr, xgb. perform one-hot encoding, use pipeline and cross validation.
"""

if client is not None:
    response3 = client.responses.create(model=OPENAI_MODEL, input=quality_prompt3)
    print('\nLLM response:')
    print(response3.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')

# Caveat: even a well-specified prompt can return unnecessarily heavy code (e.g. GridSearchCV)
# for a small, simple dataset. Always check runtime and complexity against what the task needs.

## B6. Select the model for final evaluation

**Rule enforced here — don't use the test set to choose the model.** Model *selection* uses only the
CV table (built entirely from training data). The test set is reserved for one, final, honest
evaluation of whichever model wins that selection. If you instead picked the model that scored best on
the test set, the test set would stop being a fair, unseen check — you'd have implicitly tuned to it.


In [ ]:
best_model_name = cv_results.iloc[0]['model']
best_model = dt_model
best_pred = dt_pred

if best_model_name == 'Random Forest':
    best_model = rf_model
    best_pred = rf_pred
if best_model_name == 'XGBoost':
    best_model = xgb_model
    best_pred = xgb_pred

print('Selected model:', best_model_name)
cv_results

## B7. Compare against a simple mean-price baseline

**Why this step matters:** a model is only useful if it beats the laziest possible strategy — always
guessing the average price. `DummyRegressor(strategy='mean')` implements exactly that. If a trained
model's MAE is barely better than this, the model isn't adding real value yet, however sophisticated it
looks. This baseline is why B4–B6's model comparison is not the whole evaluation story.


In [ ]:
mean_baseline = DummyRegressor(strategy='mean')
mean_baseline.fit(X_train, y_train)
mean_pred = mean_baseline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, mean_pred)
baseline_rmse = mean_squared_error(y_test, mean_pred) ** 0.5
baseline_r2 = r2_score(y_test, mean_pred)

print('Mean-price baseline')
print('MAE :', round(baseline_mae, 2))
print('RMSE:', round(baseline_rmse, 2))
print('R2  :', round(baseline_r2, 3))

In [ ]:
test_mae = mean_absolute_error(y_test, best_pred)
test_rmse = mean_squared_error(y_test, best_pred) ** 0.5
test_r2 = r2_score(y_test, best_pred)

comparison_to_baseline = pd.DataFrame({
    'model': ['Mean-price baseline', best_model_name],
    'MAE': [baseline_mae, test_mae],
    'RMSE': [baseline_rmse, test_rmse],
    'R2': [baseline_r2, test_r2]
})
comparison_to_baseline['MAE_improvement_vs_baseline'] = baseline_mae - comparison_to_baseline['MAE']
comparison_to_baseline['MAE_improvement_percent'] = (
    comparison_to_baseline['MAE_improvement_vs_baseline'] / baseline_mae * 100
)
comparison_to_baseline.round(3)

**How to interpret this table:** look at `MAE_improvement_percent` for the selected model — a
small single-digit improvement over the naive baseline is a warning sign that the features may not
carry much price signal yet; a large improvement (e.g. 40%+) is a much stronger case for deployment.


## B8. Error table with error direction

**Why direction matters, not just magnitude:** for pricing, over-predicting (quoting too high) and
under-predicting (quoting too low) usually create different business problems — lost sales versus lost
margin. A single MAE number can't tell you which failure mode dominates; this table can.


In [ ]:
error_df = X_test.copy()
error_df['actual_price'] = y_test.values
error_df['predicted_price'] = best_pred
error_df['error'] = error_df['predicted_price'] - error_df['actual_price']
error_df['absolute_error'] = error_df['error'].abs()
error_df['absolute_percentage_error'] = (error_df['absolute_error'] / error_df['actual_price']) * 100
error_df['error_direction'] = np.where(
    error_df['error'] > 0, 'over-predicted',
    np.where(error_df['error'] < 0, 'under-predicted', 'exact')
)

error_df.sort_values('absolute_error', ascending=False).head(10)

## B9. Actual vs predicted scatter, and residual plot

**Why a scatter plot in addition to a table:** a single MAE number is an *average* — it can hide
whether the model is uniformly a-bit-wrong everywhere, or spot-on for most laptops and wildly wrong for
a few expensive outliers. Points far from the diagonal line, or a widening spread at high prices, are
patterns a table of ten rows won't show you as clearly.


In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(error_df['actual_price'], error_df['predicted_price'], alpha=0.7)
min_value = min(error_df['actual_price'].min(), error_df['predicted_price'].min())
max_value = max(error_df['actual_price'].max(), error_df['predicted_price'].max())
plt.plot([min_value, max_value], [min_value, max_value])
plt.xlabel('Actual Price SGD')
plt.ylabel('Predicted Price SGD')
plt.title(f'Actual vs Predicted Price ({best_model_name})')
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(error_df['actual_price'], error_df['error'], alpha=0.7)
plt.axhline(0)
plt.xlabel('Actual Price SGD')
plt.ylabel('Prediction Error')
plt.title('Prediction Error by Actual Price')
plt.show()

error_direction_summary = error_df['error_direction'].value_counts().reset_index()
error_direction_summary.columns = ['error_direction', 'count']
error_direction_summary['percent'] = error_direction_summary['count'] / len(error_df) * 100
error_direction_summary.round(2)

**How to interpret the residual plot:** points scattered evenly above and below the zero line
suggest no systematic bias. A clear upward or downward slope in the errors as price increases suggests
the model is *systematically* mispricing one end of the market — worth investigating before trusting
the model there.


## B10. Error by price band and by brand

**Why segment the error, not just look at overall MAE:** an average error can conceal that the model
is excellent for mid-range laptops and poor for premium ones (or vice versa) — segments that matter
differently to the business. We check both **MAE** (dollar error) and **MAPE** (percentage error)
because a $200 error means very different things for a $300 laptop and a $3000 laptop.


In [ ]:
error_df['price_band'] = pd.qcut(
    error_df['actual_price'],
    q=4,
    labels=['lowest price band', 'lower-middle price band', 'upper-middle price band', 'highest price band']
)

price_band_error = error_df.groupby('price_band', observed=True).agg(
    count=('absolute_error', 'size'),
    mean_actual_price=('actual_price', 'mean'),
    mae=('absolute_error', 'mean'),
    mape=('absolute_percentage_error', 'mean'),
    median_absolute_error=('absolute_error', 'median')
).reset_index()

price_band_error.round(2)

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(price_band_error['price_band'], price_band_error['mae'])
plt.xticks(rotation=20, ha='right')
plt.ylabel('MAE')
plt.title('MAE by Price Band')
plt.show()

In [ ]:
if 'Brand' in error_df.columns:
    brand_error = error_df.groupby('Brand').agg(
        count=('absolute_error', 'size'),
        mae=('absolute_error', 'mean'),
        mape=('absolute_percentage_error', 'mean'),
        mean_actual_price=('actual_price', 'mean')
    )
    brand_error = brand_error.sort_values('mae', ascending=False)
    display(brand_error.head(10).round(2))

    # Only plot brands with at least 3 test records, so one unusual laptop
    # doesn't make a brand look far worse than it really is.
    brand_error_plot = brand_error[brand_error['count'] >= 3].sort_values('mae', ascending=False).head(12)
    plt.figure(figsize=(12, 5))
    plt.bar(brand_error_plot.index, brand_error_plot['mae'])
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('MAE by brand')
    plt.title('Largest average errors by brand')
    plt.show()
else:
    print('Brand column not found in error_df. Skip this section or replace Brand with another group column.')

**Caution when interpreting group-level error:** a brand with only 1–2 test rows can look terrible
purely by chance on one unusual laptop — hence the `count >= 3` filter above. Don't generalise from a
tiny sample.


## B11. Permutation feature importance

**Why permutation importance specifically:** it's model-agnostic and directly answers "does the model
actually rely on this column to make good predictions" — by shuffling one column at a time and
measuring how much worse the model's MAE gets. A big MAE increase after shuffling means that column was
doing real work; little to no change means the model barely uses it.

**How to interpret the output — the caveat is the most important part:** importance tells you what the
model *relies on*, not what *causes* price. A feature can be important because it's a proxy for
something else that isn't in the dataset (e.g. "screen size" might really be standing in for "laptop
category: gaming vs ultrabook"). Never present feature importance as causal evidence.


In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=10, random_state=RANDOM_STATE, scoring='neg_mean_absolute_error'
)

importance_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
})
importance_df = importance_df.sort_values('importance_mean', ascending=False)
importance_df.head(10).round(3)

In [ ]:
importance_plot = importance_df.head(10).sort_values('importance_mean')
plt.figure(figsize=(9, 5))
plt.barh(importance_plot['feature'], importance_plot['importance_mean'])
plt.xlabel('Increase in MAE after shuffling feature')
plt.title('Top 10 Permutation Importance Features')
plt.show()

## B12. LLM-assisted final judgement

### Why this prompt is used

This is the most tightly-constrained prompt in the whole notebook, on purpose. By this point we have
**real evidence** (baseline comparison, test metrics, error direction, price-band errors, brand errors,
feature importance, worst individual errors) — the prompt's job is only to help *word* a judgement from
that evidence, not to generate new claims. That's why it:

- **feeds in only the computed evidence tables**, never the raw dataset,
- says explicitly **"do not invent causes"** and **"do not overclaim"** — the two failure modes most
  likely when an LLM is asked to interpret model results: fabricating a causal story, or being more
  confident than the evidence supports,
- gives a **fixed 5-point structure** (beats baseline? / one strength / one weakness / one caution /
  one next step) so the output is a checklist you can verify line-by-line against the tables above,
  rather than free-form prose that's hard to fact-check.

**How to interpret the output:** treat it as a first draft of your written judgement, not the judgement
itself — check every claim it makes against the actual tables before using it.


In [ ]:
largest_errors = error_df[['actual_price', 'predicted_price', 'error', 'absolute_error', 'absolute_percentage_error']]
largest_errors = largest_errors.sort_values('absolute_error', ascending=False).head(5)

top_features = importance_df.head(5)

brand_section = 'Brand column not available.'
if 'Brand' in error_df.columns:
    brand_section = brand_error.head(5).round(2).to_string()

evidence_text = f"""
Task: Predict laptop price in SGD.
Selected model: {best_model_name}

Comparison against mean-price baseline:
{comparison_to_baseline.round(3).to_string(index=False)}

Selected model test performance:
Test MAE: {test_mae:.2f}
Test RMSE: {test_rmse:.2f}
Test R2: {test_r2:.3f}

Error direction summary:
{error_direction_summary.round(2).to_string(index=False)}

Error by price band:
{price_band_error.round(2).to_string(index=False)}

Top 5 brands with largest MAE:
{brand_section}

Top 5 permutation importance features:
{top_features.round(3).to_string(index=False)}

Top 5 largest individual errors:
{largest_errors.round(2).to_string(index=False)}
"""

llm_prompt = f"""
You are helping a beginner data analytics student.
Use only the evidence below to write a short model evaluation judgement.
Do not invent causes.
Do not overclaim.

Mention:
1. whether the model beats the simple baseline,
2. one strength,
3. one weakness,
4. one caution about the evidence,
5. one sensible next step.

Evidence:
{evidence_text}
"""

print(llm_prompt)

if client is not None:
    response4 = client.responses.create(model=OPENAI_MODEL, input=llm_prompt)
    print('\nLLM response:')
    print(response4.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')

**Part B checkpoint:** you should now have one selected model, a baseline comparison, held-out
test scores, at least two visual diagnostics, an error-direction summary, a price-band and brand-level
error breakdown, a permutation-importance check, and one evidence-grounded written judgement.


---
# PART C — Regression Workflow Revision (independent dataset)
*(merged from Week 5 — student performance dataset, focused on leakage)*

This part is **self-contained** and uses a different dataset (student grades) to drill a single
high-stakes idea: **a feature that was used to define the target must never be used to predict it.**


## C1. Setup


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, ShuffleSplit, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

print('Setup complete for Part C')

## C2. Load the dataset and quick checks


In [ ]:
import kagglehub

path = kagglehub.dataset_download("devansodariya/student-performance-data")
print("Downloaded to:", path)
print("Files:", os.listdir(path))

df_c = pd.read_csv(os.path.join(path, "student_data.csv"))
df_c.head()

In [ ]:
print("Shape:", df_c.shape)
print("\nColumn names:")
print(df_c.columns.tolist())
print("\nData info:")
df_c.info()

## C3. Dataset profile (payload text) for LLM review

Same design rationale as Part A's payload text: give the LLM a statistical summary rather than raw
rows, so it can flag structure, missingness, and — critically here — **correlation clues that hint at
leakage**, without needing the full dataset.


In [ ]:
from io import StringIO

buffer = StringIO()
buffer.write("=== SHAPE ===\n")
buffer.write(f"Rows: {df_c.shape[0]}, Columns: {df_c.shape[1]}\n\n")

buffer.write("=== DTYPES ===\n")
buffer.write(df_c.dtypes.to_string())
buffer.write("\n\n")

buffer.write("=== NUMERIC DESCRIBE ===\n")
buffer.write(df_c.describe().to_string())
buffer.write("\n\n")

buffer.write("=== CATEGORICAL DESCRIBE ===\n")
cat_cols_c = df_c.select_dtypes(include="object").columns.tolist()
if cat_cols_c:
    buffer.write(df_c[cat_cols_c].describe().to_string())
else:
    buffer.write("No categorical columns")
buffer.write("\n\n")

buffer.write("=== NULL SUMMARY ===\n")
null_summary = df_c.isna().sum().to_frame("null_count")
null_summary["null_pct"] = null_summary["null_count"] / len(df_c)
buffer.write(null_summary.to_string())
buffer.write("\n\n")

buffer.write("=== UNIQUE VALUES PER COLUMN ===\n")
buffer.write(df_c.nunique().to_frame("unique_count").to_string())
buffer.write("\n\n")

buffer.write("=== CORRELATIONS: NUMERIC ONLY ===\n")
buffer.write(df_c.corr(numeric_only=True).round(3).to_string())
buffer.write("\n\n")

buffer.write("=== POSSIBLE UNIQUE-ID COLUMNS ===\n")
unique_id_cols = df_c.columns[df_c.nunique() == len(df_c)].tolist()
buffer.write(str(unique_id_cols))
buffer.write("\n")

payload_text_c = buffer.getvalue()
print(payload_text_c)

## C4. Set up the API connection (Part C copy)


In [ ]:
RUN_API_CELLS = True
OPENAI_MODEL = 'gpt-5.4-nano'
client = None

if RUN_API_CELLS == True:
    from google.colab import userdata
    from openai import OpenAI
    api_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=api_key)
    print('OpenAI client is ready.')
else:
    print('Manual chatbot mode. Copy the prompts when they appear.')

### Why this prompt is used — a small, focused review question

The prompt explicitly states the **target definition** ("failed subject = grade below 10, target is the
count across G1/G2/G3") *before* asking for data risks. **Why give the target definition up front rather
than asking the LLM to guess it:** guessing invites the LLM to miss the leakage risk entirely — by
telling it exactly how the target will be built, we're asking a much sharper, checkable question:
"given this target formula, what should I be careful about?" This is a deliberately small ask (three
short sections, no modelling code yet) — asking for everything at once (cleaning + leakage + modelling
code in one prompt) tends to produce a shallower answer on each part.

**How to interpret the output:** the "columns that may cause leakage" section is the one to scrutinise
hardest — if it doesn't flag G1/G2/G3, that's a sign the model didn't fully use the target definition
you gave it, and you should trust your own reasoning (below) over its answer.


In [ ]:
llm_prompt_c = f"""
You are helping me review a regression dataset for a practical test revision.

Task:
I want to predict the number of failed subjects.
A subject is failed when its grade is below 10.
The grade columns are G1, G2 and G3.

Dataset profile:
{payload_text_c}

Please answer in three sections:
1. Data quality issues to check before modelling.
2. Columns that may cause leakage, with reasons.
3. Which recommendations need human judgement before applying.

Keep the answer concise. Do not write modelling code yet.
"""

print(llm_prompt_c)

if client is not None:
    response_c = client.responses.create(model=OPENAI_MODEL, input=llm_prompt_c)
    print('\nLLM response:')
    print(response_c.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')

**Judgement note (write this yourself, don't just accept the LLM's list):**
*I will not use G1, G2, and G3 as predictors because they directly define the target. The LLM can
suggest, but I must judge — this is the core discipline this whole notebook is testing.*


## C5. Create the target: number of failed subjects

A subject is failed when its grade is **strictly below 10** (`< 10`, not `<= 10` — this is a common
off-by-one mistake that silently changes the target's meaning).


In [ ]:
grade_cols = ["G1", "G2", "G3"]

df_c["num_failed_subjects"] = (df_c[grade_cols] < 10).sum(axis=1)

print(df_c[grade_cols + ["num_failed_subjects"]].head())
print("\nTarget value counts:")
print(df_c["num_failed_subjects"].value_counts().sort_index())
print("\nTarget proportions:")
print(df_c["num_failed_subjects"].value_counts(normalize=True).sort_index())

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df_c["num_failed_subjects"], bins=[-0.5, 0.5, 1.5, 2.5, 3.5], edgecolor="black")
plt.xticks([0, 1, 2, 3])
plt.title("Target distribution: number of failed subjects")
plt.xlabel("num_failed_subjects")
plt.ylabel("count")
plt.show()

**Why this is still treated as regression, not classification, even though the values look like
categories (0, 1, 2, 3):** the target is a *count*, and counts are ordered and additive in a way plain
categories aren't (2 failed subjects is "more" than 1, in a way that "red" isn't more than "blue"). If
the task instead asked "did the student fail at least one subject?" (yes/no), that would become
classification — the modelling approach follows from how the target is *defined*, not from how many
distinct values it happens to take.


## C6. Define X and y, then remove leakage columns

**Why G1, G2, G3 must be dropped, explained concretely:** `num_failed_subjects` is *calculated
directly* from these three columns. If they're left in `X`, the model doesn't need to learn anything
about student behaviour or background — it can just re-derive the count from the grades it's literally
being handed. That's leakage: the model would score extremely well in testing and then be useless the
moment it needs to predict from data where you don't already know the final grades (which is the whole
point of *predicting* in the first place).


In [ ]:
target_col = "num_failed_subjects"
leakage_cols = ["G1", "G2", "G3"]

X_c = df_c.drop(columns=[target_col] + leakage_cols)
y_c = df_c[target_col]

print("X shape:", X_c.shape)
print("y shape:", y_c.shape)
print("Columns removed for leakage:", leakage_cols)

## C7. Train/test split (stratified)

**Why `stratify=y` here, when Part A/B did not stratify:** `num_failed_subjects` is a small integer
count (0–3) that behaves partly like a group label. Stratifying keeps the proportion of 0/1/2/3-failure
students similar in both train and test sets, avoiding a split where, say, the test set happens to have
almost no 3-failure students by chance. **Caveat:** if a target value is very rare, stratified splitting
can fail or produce unstable tiny groups — always check the value counts (as done in C5) before relying
on it.


In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c, y_c,
    test_size=0.10,
    random_state=42,
    stratify=y_c
)

print("Train shape:", X_train_c.shape, y_train_c.shape)
print("Test shape:", X_test_c.shape, y_test_c.shape)

print("\nTrain target proportions:")
print(y_train_c.value_counts(normalize=True).sort_index())
print("\nTest target proportions:")
print(y_test_c.value_counts(normalize=True).sort_index())

## C8. Preprocessing pipeline


In [ ]:
num_cols_c = X_train_c.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_c2 = X_train_c.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor_c = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols_c),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols_c2)
    ]
)

## C9. Train and compare models using CV

**Why Random Forest and XGBoost specifically here (and not a Decision Tree baseline):** by this point in
the course the workflow shifts from "build a baseline" (Part A/B's job) to "compare two solid,
commonly-used tabular models fairly" — which is the exact skill a practical test is checking. Both are
compared under the same split, same preprocessing, same CV scheme, same metric, so any difference in
score is attributable to the model itself.

We also record **train MAE alongside validation MAE** specifically to catch overfitting: a model that's
excellent on training data but much worse on validation data (`train_val_gap` large) has memorised
rather than generalised.


In [ ]:
cv_c = ShuffleSplit(n_splits=5, test_size=0.20, random_state=42)

models_c = {
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=8, random_state=42, n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        objective="reg:squarederror", random_state=42, n_jobs=-1
    )
}

results_c = []
fitted_models_c = {}

for name, model in models_c.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor_c),
        ("regressor", model)
    ])

    cv_out = cross_validate(
        pipe, X_train_c, y_train_c, cv=cv_c,
        scoring="neg_mean_absolute_error",
        return_train_score=True, n_jobs=-1
    )

    train_mae = -cv_out["train_score"]
    val_mae = -cv_out["test_score"]

    pipe.fit(X_train_c, y_train_c)
    y_pred_c = pipe.predict(X_test_c)
    holdout_mae = mean_absolute_error(y_test_c, y_pred_c)
    rmse_c = np.sqrt(mean_squared_error(y_test_c, y_pred_c))
    r2_c = r2_score(y_test_c, y_pred_c)

    results_c.append({
        "model": name,
        "cv_train_mae_mean": train_mae.mean(),
        "cv_train_mae_std": train_mae.std(),
        "cv_val_mae_mean": val_mae.mean(),
        "cv_val_mae_std": val_mae.std(),
        "train_val_gap": val_mae.mean() - train_mae.mean(),
        "holdout_mae": holdout_mae,
        "holdout_rmse": rmse_c,
        "holdout_r2": r2_c
    })
    fitted_models_c[name] = pipe

results_df_c = pd.DataFrame(results_c).sort_values("cv_val_mae_mean")
results_df_c

**How to interpret this table, in order:**
1. Which model has the lower `cv_val_mae_mean`? — that's the primary comparison.
2. Is `cv_train_mae_mean` much lower than `cv_val_mae_mean`? A large gap signals overfitting.
3. Does `holdout_mae` roughly agree with the CV result? If not, investigate — one of the two estimates
   may be unreliable (e.g. small test set).
4. Is the difference between models *meaningful*, or a fraction of a failed-subject count that wouldn't
   change any real decision? Don't pick the fancier-sounding model for a marginal gain.


## C10. Plot actual vs predicted values

**Why non-integer predictions are expected and fine:** the target is 0–3 failed subjects, but regression
models output continuous numbers (e.g. 1.4). That's normal — round only if the task specifically asks
for whole-number outputs.


In [ ]:
best_model_name_c = results_df_c.iloc[0]["model"]
best_model_c = fitted_models_c[best_model_name_c]
print("Best model based on CV validation MAE:", best_model_name_c)

y_pred_test_c = best_model_c.predict(X_test_c)

order = np.argsort(y_test_c.to_numpy())
y_true_sorted = y_test_c.to_numpy()[order]
y_pred_sorted = y_pred_test_c[order]

plt.figure(figsize=(10, 5))
plt.plot(y_true_sorted, label="Actual", marker="o")
plt.plot(y_pred_sorted, label="Predicted", marker="x")
plt.title(f"Actual vs Predicted: {best_model_name_c}")
plt.xlabel("Test sample index sorted by actual target")
plt.ylabel("Number of failed subjects")
plt.legend()
plt.tight_layout()
plt.show()

**How to interpret this plot:** because it's sorted by the actual value, a well-fitted model
produces a predicted line that roughly tracks the staircase shape of the actual line. Look specifically
for: does the model over-predict students with 0 failures (false alarms)? Does it under-predict
students with 2–3 failures (missed at-risk students)? Errors concentrated at the rare target values
(2 and 3) are common, since the model has fewer examples to learn from there.


## C11. Final judgement — write-up template

Use this structure for a short written response (the discipline here mirrors Part B's LLM judgement
prompt, but you're writing it yourself this time):

> I treated this as a regression task because the target is a numeric count: number of failed subjects.
> I created the target by counting how many of G1, G2, and G3 are below 10.
> I removed G1, G2, and G3 from the predictors because they directly define the target and would cause leakage.
> I used a pipeline so that preprocessing and modelling are applied consistently.
> Based on the CV validation MAE, train-validation gap, and holdout MAE, I would choose ___ because ___.

**Recap:** good practical-test answers are short, evidence-based, and clear. You do not need to explain
every line of code — you need to show you know *why* each major decision (target definition, leakage
removal, model comparison method) was made.
